# MOSS-Image Colab Studio — 開發驗證入口

目前 Stage 0/1 已完成，正式五分頁工作台開發中。這份 Notebook 啟動**真實單步載入與通訊探測**，不能當作 8/50 步畫質驗收。

先選「執行階段 → 變更執行階段類型 → L4 GPU、 大量 RAM」。不自動購買方案或掛載 Google Drive。
模型約 20 GiB，儲存在 Colab 本地。VM 回收會遺失本地檔案。入口以單次密碼保護，Quick Tunnel 使用同源 HTTP 輪詢，`share=False`，ComfyUI 只綁 loopback。

In [ ]:
# @title 1. 取得程式（開發中 main；可指定已知 commit 重演）
SOURCE_REVISION = "main" # @param {type:"string"}
import os, sys, subprocess, pathlib, json
SOURCE = pathlib.Path('/content/MOSS-Image-Colab')
if not (SOURCE / '.git').exists():
    subprocess.run(['git','clone','https://github.com/Openclaw-Metis/MOSS-Image-Colab.git',str(SOURCE)],check=True)
if subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=SOURCE,text=True).strip():
    raise RuntimeError('原始碼有本地修改，保留現況；請先另存後再重新啟動。')
subprocess.run(['git','fetch','origin',SOURCE_REVISION],cwd=SOURCE,check=True)
subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=SOURCE,check=True)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=SOURCE,text=True).strip()
sys.path.insert(0,str(SOURCE/'src'))
os.environ['PYTHONPATH']=str(SOURCE/'src')
print('Resolved source commit:',SOURCE_COMMIT)

In [ ]:
# @title 2. 檢查 L4、安裝固定環境、下載並驗證必要模型
from moss_studio.services.bootstrap import ColabBootstrap
bootstrap = ColabBootstrap(SOURCE)
print(json.dumps(__import__('moss_studio.services.resources',fromlist=['inspect_runtime']).inspect_runtime(bootstrap.root),indent=2))
bootstrap.prepare(allow_candidate=True)
bootstrap.download(['sft_bf16','qwen_int8','vae_small','edit_sft'])
engine_state=bootstrap.start_engine()
print('ComfyUI:',engine_state)

In [ ]:
# @title 3. 準備可重散布的探測素材與工作流
from PIL import Image, ImageDraw
probe_graphs={task:json.loads((SOURCE/f'workflows/probes/{name}_load.json').read_text()) for task,name in [('text','t2i'),('vision','vision'),('edit','edit')]}
(bootstrap.root/'probe-graphs.json').write_text(json.dumps(probe_graphs))
reference=Image.new('RGB',(512,512),'#f0e0c0'); drawing=ImageDraw.Draw(reference)
drawing.rectangle((72,120,280,400),fill='#d02030'); drawing.ellipse((300,80,460,240),fill='#2050b0')
reference.save(bootstrap.comfy/'input/stage1_reference.png')

In [ ]:
# @title 4. 啟動已認證的 Gradio 輪詢診斷頁
import secrets, urllib.request
from moss_studio.services.processes import ManagedProcess
if 'studio_password' not in globals():
    studio_password=secrets.token_urlsafe(24)
os.environ.update(MOSS_PROBE_PASSWORD=studio_password, MOSS_PROBE_ROOT=str(bootstrap.root),
                  GRADIO_ANALYTICS_ENABLED='False', GRADIO_TEMP_DIR=str(bootstrap.root/'gradio_cache'))
probe_app=ManagedProcess(bootstrap.root/'.runtime','probe_app')
def app_health():
    with urllib.request.urlopen('http://127.0.0.1:7860/login',timeout=3) as response:
        if response.status != 200: raise RuntimeError('App not ready')
try:
    app_state=probe_app.start([str(bootstrap.python),str(SOURCE/'scripts/polling_probe.py')],SOURCE,7860,app_health)
finally:
    os.environ.pop('MOSS_PROBE_PASSWORD',None)
print('App PID:',app_state['pid'],'reused:',app_state['reused'])
print('本次入口密碼：',studio_password)
print('這是暫時密碼；不要儲存或分享含密碼的 Notebook 輸出。')

In [ ]:
# @title 5. 啟動固定版本 Quick Tunnel（不使用 Gradio share）
import hashlib, re
runtime=json.loads((SOURCE/'configs/runtime.lock.json').read_text())
binary=bootstrap.root/'cloudflared'; pinned=runtime['cloudflared']
if not binary.exists() or hashlib.sha256(binary.read_bytes()).hexdigest()!=pinned['sha256']:
    with urllib.request.urlopen(pinned['url'],timeout=60) as response: data=response.read(100*1024**2+1)
    if len(data)>100*1024**2 or hashlib.sha256(data).hexdigest()!=pinned['sha256']:
        raise RuntimeError('cloudflared 校驗失敗，不啟動')
    partial=binary.with_suffix('.part'); partial.write_bytes(data); partial.chmod(0o700); partial.replace(binary)
tunnel=ManagedProcess(bootstrap.root/'.runtime','tunnel')
def tunnel_health():
    with urllib.request.urlopen('http://127.0.0.1:20242/ready',timeout=3) as response:
        if response.status!=200: raise RuntimeError('Tunnel not ready')
state=tunnel.start([str(binary),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate','--protocol','http2','--metrics','127.0.0.1:20242'],SOURCE,20242,tunnel_health)
urls=re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com',tunnel.log.read_text())
if not urls: raise RuntimeError('通道未回報網址；不退回 share=True')
print('Gradio 輪詢診斷入口：',urls[-1])
print('請用上一格產生的單次密碼登入。此頁尚非完整五分頁產品。')

## 操作與關閉

`text` 測文生圖載入；`vision` 將上傳圖送進視覺塔；`edit` 使用 Edit LoRA。單步探測結果可能粗糙。
成功後可下載探測 ZIP；重新載入頁面會查找原工作，不會自動重跑。

需要停止時，先等工作終止，再執行下方儲存格。只停止本工具擁有的程序，保留模型與作品。

In [ ]:
# @title 6. 停止本工具（按需單獨執行；預設不停止）
STOP_STUDIO = False # @param {type:"boolean"}
if STOP_STUDIO:
    for name in ['tunnel','probe_app']:
        if name in globals(): globals()[name].stop()
    bootstrap.engine.stop()
    print('已停止本工具程序，未刪除作品或模型。')